In [40]:
import os
import sys


CURRENT_DIR = os.getcwd()
FOLDERS = CURRENT_DIR.split(os.sep)
TESIS_FOLDER_INDEX = FOLDERS.index('S-noise-gradient')
CURRENT_DIR = os.sep.join(FOLDERS[:TESIS_FOLDER_INDEX+1])
LIBS_PATH = os.path.join(CURRENT_DIR, 'src', 'libs')
assert os.path.exists(LIBS_PATH)
sys.path.append(CURRENT_DIR)
sys.path.append(LIBS_PATH)


# Import external and local libraries
# __________________________________________________________________________________________________
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
import json
from typing import List

# Locals
from utils import *
from dataset import DFDataset, TensorDataset
from models import *
import config as cfg
from dlnr import DLNoiseReduction


# Global / Path variables
# __________________________________________________________________________________________________
DATA_PATH = os.path.join(CURRENT_DIR, 'data', 'try_ts')
CHECKPOINT_PATH = os.path.join(CURRENT_DIR, 'checkpoints')
CONFIG_PATH = os.path.join(CURRENT_DIR, 'config')


# Make sure that the GPU is being used
# __________________________________________________________________________________________________
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert device.type == "cuda"

In [41]:
df = pd.read_parquet(os.path.join(DATA_PATH, 'electric_motor_temperature', 'measures_v2.parquet'))
df

,u_q,coolant,stator_winding,u_d,stator_tooth,motor_speed,i_d,i_q,pm,stator_yoke,ambient,torque,profile_id
0,-0.450682,18.805172,19.086670,-0.350055,18.293219,0.002866,0.004419,0.000328,24.554214,18.316547,19.850691,1.871008e-01,17
1,-0.325737,18.818571,19.092390,-0.305803,18.294807,0.000257,0.000606,-0.000785,24.538078,18.314955,19.850672,2.454175e-01,17
2,-0.440864,18.828770,19.089380,-0.372503,18.294094,0.002355,0.001290,0.000386,24.544693,18.326307,19.850657,1.766153e-01,17
3,-0.327026,18.835567,19.083031,-0.316199,18.292542,0.006105,0.000026,0.002046,24.554018,18.330833,19.850647,2.383027e-01,17
4,-0.471150,18.857033,19.082525,-0.332272,18.291428,0.003133,-0.064317,0.037184,24.565397,18.326662,19.850639,2.081967e-01,17
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1330811,-2.152128,30.721298,45.274497,0.791801,41.934347,0.000380,-2.000169,1.097528,62.147780,38.653720,23.989078,-8.116730e-60,71
1330812,-2.258684,30.721306,45.239017,0.778900,41.868923,0.002985,-2.000499,1.097569,62.142646,38.656328,23.970700,-5.815891e-60,71
1330813,-2.130312,30.721312,45.211576,0.804914,41.804819,0.002301,-1.999268,1.098765,62.138387,38.650923,23.977234,-4.167268e-60,71
1330814,-2.268498,30.721316,45.193508,0.763091,41.762220,0.005662,-2.000999,1.095696,62.133422,38.655686,24.001421,-2.985978e-60,71


In [42]:
nan_counts = df.isna().sum()
print((nan_counts/len(df))*100)

u_q               0.0
coolant           0.0
stator_winding    0.0
u_d               0.0
stator_tooth      0.0
motor_speed       0.0
i_d               0.0
i_q               0.0
pm                0.0
stator_yoke       0.0
ambient           0.0
torque            0.0
profile_id        0.0
dtype: float64


In [43]:
np.unique(df['profile_id'].values, return_counts=True)

(array([ 2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18,
        19, 20, 21, 23, 24, 26, 27, 29, 30, 31, 32, 36, 41, 42, 43, 44, 45,
        46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62,
        63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 78, 79, 80,
        81]),
 array([19357, 19248, 33424, 14788, 40388, 14651, 18757, 20336, 15256,
         7887, 21942, 35906, 18598, 18124, 20645, 15964, 37732, 10410,
        43971, 17321, 11856, 15015, 16666, 35361, 21358, 23863, 15587,
        20960, 22609, 16700, 16920,  8443, 26341, 17142,  2180,  2176,
        21983, 10816, 10810,  6261,  3726, 32442, 10807, 10807, 33123,
        14403, 33382,  7475, 14543, 14516, 25600, 16668,  6250, 40094,
        36476, 11135, 23331, 15350, 25677, 14656, 15301, 16786, 23761,
        13472, 22188,  8445, 31154, 23824, 17672]))

In [44]:
# Calcular el número de outliers para cada columna numérica
outliers_percentage = {}
for column in df.select_dtypes(include=[np.number]).columns:
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers_percentage[column] = f'{round(((df[column] < lower_bound) | (df[column] > upper_bound)).sum()/len(df) * 100, 4)} %'

# Mostrar el número de outliers por columna
outliers_percentage

{'u_q': '0.0 %',
 'coolant': '0.0002 %',
 'stator_winding': '0.0 %',
 'u_d': '1.4181 %',
 'stator_tooth': '0.0 %',
 'motor_speed': '0.0 %',
 'i_d': '0.0 %',
 'i_q': '4.9316 %',
 'pm': '0.0 %',
 'stator_yoke': '0.0 %',
 'ambient': '0.0195 %',
 'torque': '3.2657 %',
 'profile_id': '0.0 %'}

In [45]:
df['y'] = df['pm']
df.drop(columns=['pm'], inplace=True)

In [47]:
# Separate the DataFrame into different datasets based on 'profile_id'
datasets = {profile_id: group for profile_id, group in df.groupby('profile_id')}
datasets = {profile_id: group.drop(columns=['profile_id']) for profile_id, group in datasets.items()}

In [48]:
# Check if indices are consecutive for each dataset
consecutive_indices = {}
for profile_id, dataset in datasets.items():
    is_consecutive = (dataset.index.to_series().diff().dropna() == 1).all()
    consecutive_indices[profile_id] = is_consecutive

# Display the result
any(value == False for value in consecutive_indices.values())

False

In [50]:
for i, (profile_id, dataset) in enumerate(datasets.items()):
    filename = f"clean{'_' + str(profile_id) if i > 0 else ''}.parquet"
    dataset.to_parquet(os.path.join(DATA_PATH, 'electric_motor_temperature', filename))